# Batch variant scoring

In this notebook, we demonstrate how to score many variants using AlphaGenome.

To prepare numerous variants for scoring, provide the following information as
columns in a tab-separated Variant Call Format (VCF) file: - `variant_id`: A
unique identifier for each variant. - `CHROM`: Chromosome, specified as a string
beginning with 'chr' (e.g., 'chr1'). - `POS`: Integer representing the base pair
position (1-based; build hg38 (human) or mm10 (mouse) - see
[FAQ](https://www.alphagenomedocs.com/faqs.html#how-do-i-define-a-variant)). -
`REF`: The reference nucleotide sequence at the specified position. - `ALT`: The
alternate nucleotide sequence at the specified position.

``` {tip}
Open this tutorial in Google Colab for interactive viewing.
```

In [1]:
# @title Install AlphaGenome

# @markdown Run this cell to install AlphaGenome.
from IPython.display import clear_output
! pip install alphagenome
clear_output()

In [2]:
# @title Setup and imports.

from io import StringIO
from alphagenome import colab_utils
from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers
from google.colab import data_table, files
import pandas as pd
from tqdm import tqdm

data_table.enable_dataframe_formatter()

# Load the model.
dna_model = dna_client.create(colab_utils.get_api_key())

In [5]:
# @title Score batch of variants.

# Load VCF file containing variants.
vcf_file = '/content/AQP4_all_variants_MAF.vcf'  # @param

# We provide an example list of variants to illustrate.
#vcf_file = """variant_id\tCHROM\tPOS\tREF\tALT
#chr3_58394738_A_T_b38\tchr3\t58394738\tA\tT
#chr8_28520_G_C_b38\tchr8\t28520\tG\tC
#chr16_636337_G_A_b38\tchr16\t636337\tG\tA
#chr16_1135446_G_T_b38\tchr16\t1135446\tG\tT
#"""

vcf_full = pd.read_csv(vcf_file, sep='\t')

required_columns = ['variant_id', 'CHROM', 'POS', 'REF', 'ALT']
for column in required_columns:
  if column not in vcf_full.columns:
    raise ValueError(f'VCF file is missing required column: {column}.')

organism = 'human'  # @param ["human", "mouse"] {type:"string"}

# @markdown Specify length of sequence around variants to predict:
sequence_length = '1MB'  # @param ["16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]

# @markdown Specify which scorers to use to score your variants:
score_rna_seq = True  # @param { type: "boolean"}
score_cage = True  # @param { type: "boolean" }
score_procap = True  # @param { type: "boolean" }
score_atac = True  # @param { type: "boolean" }
score_dnase = True  # @param { type: "boolean" }
score_chip_histone = True  # @param { type: "boolean" }
score_chip_tf = True  # @param { type: "boolean" }
score_polyadenylation = True  # @param { type: "boolean" }
score_splice_sites = True  # @param { type: "boolean" }
score_splice_site_usage = True  # @param { type: "boolean" }
score_splice_junctions = True  # @param { type: "boolean" }

# @markdown Other settings:
download_predictions = True  # @param { type: "boolean" }

# Parse organism specification.
organism_map = {
    'human': dna_client.Organism.HOMO_SAPIENS,
    'mouse': dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[organism]

# Parse scorer specification.
scorer_selections = {
    'rna_seq': score_rna_seq,
    'cage': score_cage,
    'procap': score_procap,
    'atac': score_atac,
    'dnase': score_dnase,
    'chip_histone': score_chip_histone,
    'chip_tf': score_chip_tf,
    'polyadenylation': score_polyadenylation,
    'splice_sites': score_splice_sites,
    'splice_site_usage': score_splice_site_usage,
    'splice_junctions': score_splice_junctions,
}

all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
selected_scorers = [
    all_scorers[key]
    for key in all_scorers
    if scorer_selections.get(key.lower(), False)
]

# Remove any scorers or output types that are not supported for the chosen organism.
unsupported_scorers = [
    scorer
    for scorer in selected_scorers
    if (
        organism.value
        not in variant_scorers.SUPPORTED_ORGANISMS[scorer.base_variant_scorer]
    )
    | (
        (scorer.requested_output == dna_client.OutputType.PROCAP)
        & (organism == dna_client.Organism.MUS_MUSCULUS)
    )
]
if len(unsupported_scorers) > 0:
  print(
      f'Excluding {unsupported_scorers} scorers as they are not supported for'
      f' {organism}.'
  )
  for unsupported_scorer in unsupported_scorers:
    selected_scorers.remove(unsupported_scorer)



#if 'df_scores' not in locals() and 'df_scores' not in globals():
df_scores = pd.DataFrame()

chunk_size = 100  # number of rows per chunk
print(len(vcf_full)/chunk_size, "chunks")

for i in range(0, len(vcf_full), chunk_size):
  print("Processing chunk", i/chunk_size)
  vcf = vcf_full.iloc[i:i + chunk_size,:]

  # Score variants in the VCF file.
  results = []

  for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    variant = genome.Variant(
       chromosome=str(vcf_row.CHROM),
       position=int(vcf_row.POS),
       reference_bases=vcf_row.REF,
       alternate_bases=vcf_row.ALT,
       name=vcf_row.variant_id,
    )
    interval = variant.reference_interval.resize(sequence_length)

    variant_scores = dna_model.score_variant(
        interval=interval,
       variant=variant,
       variant_scorers=selected_scorers,
        organism=organism,
   )
    results.append(variant_scores)

  df_scores_part = variant_scorers.tidy_scores(results)
  df_scores_part = df_scores_part.loc[(df_scores_part['quantile_score'] > 0.90) | (df_scores_part['quantile_score'] < -0.90)]
  df_scores_part = df_scores_part.loc[(df_scores_part['biosample_name'] == 'astrocyte') | (df_scores_part['biosample_name'] == 'brain')]
  df_scores = pd.concat([df_scores, df_scores_part], ignore_index=True)

if download_predictions:
  df_scores.to_csv('variant_scores.csv', index=False)
  files.download('variant_scores.csv')

df_scores

0.36 chunks
Processing chunk 0.0


100%|██████████| 36/36 [00:47<00:00,  1.33s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,...,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score,quantile_score
0,chr18:26861589:G>A,chr18:26337301-27385877:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,...,tissue,embryonic,encode,single,False,NaN,H3K4me3,NaN,-0.020830,-0.923901
1,chr18:26861589:G>A,chr18:26337301-27385877:.,ENSG00000266184,ENSG00000266184,processed_pseudogene,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,-0.003205,-0.905090
2,chr18:26855354:G>A,chr18:26331066-27379642:.,None,None,None,None,None,None,CAGE,"CenterMaskScorer(requested_output=CAGE, width=...",...,tissue,NaN,fantom,NaN,NaN,NaN,NaN,NaN,-0.044091,-0.911155
3,chr18:26855354:G>A,chr18:26331066-27379642:.,ENSG00000266184,ENSG00000266184,processed_pseudogene,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,0.003167,0.905090
4,chr18:26855354:G>A,chr18:26331066-27379642:.,ENSG00000171885,AQP4,protein_coding,-,None,None,RNA_SEQ,PolyadenylationScorer(),...,primary_cell,unknown,encode,paired,False,NaN,NaN,,0.031858,0.985213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,chr18:26866755:T>C,chr18:26342467-27391043:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,...,tissue,embryonic,encode,single,False,NaN,H3K4me3,NaN,0.023496,0.933411
262,chr18:26866755:T>C,chr18:26342467-27391043:.,None,None,None,None,None,None,CAGE,"CenterMaskScorer(requested_output=CAGE, width=...",...,tissue,NaN,fantom,NaN,NaN,NaN,NaN,NaN,0.086881,0.970426
263,chr18:26855248:G>T,chr18:26330960-27379536:.,ENSG00000171885,AQP4,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,-0.004270,-0.949106
264,chr18:26855248:G>T,chr18:26330960-27379536:.,ENSG00000265369,PCAT18,lncRNA,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,-0.004017,-0.941769


In [10]:
df_scores = pd.read_csv('variant_scores_MAF.csv')


Note that the resulting output dataframe could be quite large, especially if you
use many scorers for scoring. Very large dataframes can't be filtered
interactively, but you can interact with them using the standard `pandas`
commands:

In [15]:
# Examine just the effects of the variants on T-cells.
#columns = [c for c in df_scores.columns if c != 'ontology_curie']
#df_scores[(df_scores['ontology_curie'] == 'CL:0000127')][columns]

df_scores = df_scores.loc[(df_scores['quantile_score'] > 0.95) | (df_scores['quantile_score'] < -0.95)]
df_scores = df_scores.loc[df_scores['biosample_name'] == 'astrocyte']
#df_scores = df_scores.loc[df_scores['biosample_name'] == 'brain']
df_scores['SNP'] = df_scores['variant_id']
df_scores['Main MAF Population'] = df_scores['variant_id']

df_scores = df_scores.replace({"SNP":
    {'chr18:26855354:G>A': 'rs114892361',
     'chr18:26860169:C>A': 'rs114622499',
     'chr18:26863064:C>G': 'rs112481323',
     'chr18:26863655:A>G': 'rs111931765',
     'chr18:26863111:G>A': 'rs77905661',
     'chr18:26857815:G>A': 'rs73945980',
     'chr18:26857134:G>A': 'rs73945979',
     'chr18:26854566:C>G': 'rs73945976',
     'chr18:26863088:G>A': 'rs72878776',
     'chr18:26862263:C>T': 'rs72557968',
     'chr18:26864186:T>C': 'rs12968026',
     'chr18:26865469:T>C': 'rs3875089',
     'chr18:26859108:A>G': 'rs335931',
     'chr18:26864250:G>A': 'rs162009',
     'chr18:26867822:G>A': 'rs162005',
     }})

df_scores = df_scores.replace({"Main MAF Population":
    {'chr18:26855354:G>A': 'African',
     'chr18:26860169:C>A': 'African',
     'chr18:26863064:C>G': 'African',
     'chr18:26863655:A>G': 'African',
     'chr18:26863111:G>A': 'European, African',
     'chr18:26857815:G>A': 'African',
     'chr18:26857134:G>A': 'African',
     'chr18:26854566:C>G': 'African, Latin American 1',
     'chr18:26863088:G>A': 'European',
     'chr18:26862263:C>T': 'South Asian',
     'chr18:26864186:T>C': 'European',
     'chr18:26865469:T>C': 'European, African, Latin American 1, South Asian',
     'chr18:26859108:A>G': 'European, Asian, Latin American',
     'chr18:26864250:G>A': 'European, African, Latin American, Asian',
     'chr18:26867822:G>A': 'European, African, Latin American, Asian (ALT > REF)',
     }})

cols_of_interest = ['variant_id','SNP','gene_name','gene_type','output_type','variant_scorer','raw_score','quantile_score','junction_Start',
                    'junction_End','transcription_factor','histone_mark','ontology_curie','biosample_name', 'Main MAF Population']
df_scores[cols_of_interest]
#df_scores


,variant_id,SNP,gene_name,gene_type,output_type,variant_scorer,raw_score,quantile_score,junction_Start,junction_End,transcription_factor,histone_mark,ontology_curie,biosample_name,Main MAF Population
4,chr18:26855354:G>A,rs114892361,AQP4,protein_coding,RNA_SEQ,PolyadenylationScorer(),0.031858,0.985213,NaN,NaN,NaN,NaN,CL:0000127,astrocyte,African
6,chr18:26860169:C>A,rs114622499,AQP4,protein_coding,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),-0.003705,-0.962893,NaN,NaN,NaN,NaN,CL:0000127,astrocyte,African
14,chr18:26863064:C>G,rs112481323,NaN,NaN,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,0.045665,0.967613,NaN,NaN,NaN,H3K27me3,CL:0000127,astrocyte,African
20,chr18:26863064:C>G,rs112481323,AQP4,protein_coding,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),-0.006757,-0.993767,NaN,NaN,NaN,NaN,CL:0000127,astrocyte,African
23,chr18:26863064:C>G,rs112481323,AQP4,protein_coding,SPLICE_JUNCTIONS,SpliceJunctionScorer(),0.047485,0.979689,26862596.0,26862961.0,NaN,NaN,CL:0000127,astrocyte,African
31,chr18:26863655:A>G,rs111931765,NaN,NaN,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,0.038219,0.961175,NaN,NaN,NaN,H2AFZ,CL:0000127,astrocyte,African
32,chr18:26863655:A>G,rs111931765,NaN,NaN,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,-0.029097,-0.951345,NaN,NaN,NaN,H3K27me3,CL:0000127,astrocyte,African
33,chr18:26863655:A>G,rs111931765,NaN,NaN,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,0.060750,0.950237,NaN,NaN,NaN,H3K4me2,CL:0000127,astrocyte,African
34,chr18:26863655:A>G,rs111931765,NaN,NaN,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,0.051119,0.972998,NaN,NaN,NaN,H3K4me3,CL:0000127,astrocyte,African
35,chr18:26863655:A>G,rs111931765,NaN,NaN,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,0.039170,0.955539,NaN,NaN,NaN,H3K9ac,CL:0000127,astrocyte,African
